# Séance 4 (Notebook C) : PageRank et marche aléatoire sur graphe

**Enseignant :** Jean Delpech

**Cours :** Algorithmie et développement dans l'ingénierie des données

**Classe :** M1 Data

**Année scolaire :** 2025/2026

**Dernière mise à jour :** juin 2026

## Objectifs

- Comprendre la marche aléatoire sur un graphe et son lien avec l'importance des nœuds
- Maîtriser le principe de PageRank et son algorithme par itération de puissance
- Comprendre le rôle du facteur d'amortissement et les conditions de convergence
- Percevoir comment PageRank se généralise aux systèmes de recommandation (RWR)

## Plan

| # | Bloc |
|---|---|
| 1 | Marche aléatoire : intuition et formalisation |
| 2 | PageRank : formule et algorithme |
| 3 | Facteur d'amortissement et convergence |
| 4 | Random Walk with Restart : extension recommandation |
| 5 | Pour aller plus loin : synthèse data science |

> **Notebooks de cette séance**
> - **Notebook A** : Graphes, BFS, DFS, Backtracking, DAG
> - **Notebook B** : Dijkstra et A* (plus courts chemins pondérés)

> **Note pédagogique**
> Ce notebook est volontairement théorique. Les algorithmes sont présentés sous forme de pseudo-code commenté. L'implémentation *from scratch* constitue le travail attendu dans le cadre des mini-mémoires.

# PARTIE 1 : Marche aléatoire sur un graphe

## 1.1 Intuition

Imaginons une personne qui navigue aléatoirement sur des pages web: à chaque page, elle clique sur un lien au hasard parmi ceux disponibles. Après un très grand nombre de clics, quelle proportion du temps passera-t-elle sur chaque page ?

Cette proportion reflète l'**importance** de la page. Une page vers laquelle beaucoup d'autres pages pointent sera visitée souvent, non pas parce qu'on la cible directement, mais parce que les visiteurs y arrivent naturellement depuis de nombreux chemins différents.
C'est exactement ce que mesure **PageRank**. Pour appréhender le concept, commençons par découvrir celui de **marche aléatoire** (*random walk*).

## 1.2 Définition formelle

Une **marche aléatoire** sur un graphe orienté G = (V, E) est définie ainsi :

- À l'étape t, le visiteur est sur le nœud u
- Il choisit **uniformément au hasard** l'un des arcs sortants de u
- Il se déplace vers le nœud v correspondant (avec probabilité 1 / out_degree(u))
- Il répète indéfiniment

La **matrice de transition** T encode ces probabilités :

```
T[v][u] = 1 / out_degree(u)   si l'arc u → v existe
T[v][u] = 0                   sinon
```

L'état de la marche à l'étape t est un vecteur de probabilités p(t), où p(t)[v] est la probabilité d'être sur le nœud v à l'étape t. La dynamique est simplement :

```
p(t+1) = T × p(t)
```

## 1.3 Distribution stationnaire

Après suffisamment de pas (suffisamment longtemps), le vecteur p(t) converge vers une **distribution stationnaire** π qui ne change plus d'une étape à l'autre :

```
π = T × π
```
> c’est une notation classique des [chaînes de Markov](https://fr.wikipedia.org/wiki/Cha%C3%AEne_de_Markov)

Cette distribution stationnaire satisfait, pour chaque nœud v :

$$
\pi(v) = \sum_{\substack{u \in V \\ u \rightarrow v}} \frac{\pi(u)}{\text{out\_degree}(u)}
$$

Le $\sum$ vient de ce qu’on fait la somme sur tous les nœuds $u$ qui ont un arc vers $v$.
Interprétation : la probabilité à long terme d'être en $v$ est la somme des probabilités de venir de chaque voisin entrant $u$, pondérée par la probabilité de choisir l'arc $u$→$v$.

## 1.4 Deux problèmes qui brisent la convergence

La convergence vers une distribution stationnaire unique n'est garantie que si le graphe est **fortement connexe** (on peut aller de tout nœud vers tout autre nœud) et **apériodique**.

En pratique, deux cas posent problème :

**Dangling nodes** : ce sont des nœuds sans arc sortant (culs-de-sac)

```
   A → B → C
              (aucun arc sortant depuis C)
```

Quand le surfeur atteint C, il est bloqué. La probabilité s'accumule sur C et ne se redistribue plus. La marche n'a plus de distribution stationnaire bien définie.

**Composantes déconnectées** : sous-graphes sans lien entre eux

```
   A  →  B          C  →  D
   ↑_____|          ↑_____|
   (composante 1)   (composante 2)
```

Selon le nœud de départ, le viste reste piégé dans sa composante. La distribution stationnaire dépend alors du nœud initial, elle n'est plus unique.

**PageRank résout les deux problèmes avec le facteur d'amortissement**, présenté en partie 2.

# PARTIE 2 : PageRank (formule et algorithme)

## 2.1 La formule PageRank

PageRank modifie la marche aléatoire avec un **facteur d'amortissement** d ∈ ]0, 1[ :

$$
\text{PR}(v) = \frac{1-d}{N} + d \sum_{\substack{u \in V \\ u \rightarrow v}} \frac{\text{PR}(u)}{\text{out\_degree}(u)}
$$

- on reprend ici la notation de Brin & Page (1998) qui dans leur article notent PR(u) plutôt que π(u), mais c’est la même chose.
- **N** : nombre total de nœuds
- **(1 - d) / N** : « *téléportation* », avec probabilité (1-d), le visiteur abandonne sa navigation et saute vers un nœud **complètement aléatoire** (téléportation uniforme)
- **d × Σ ...** : *contribution des lien entrants*, avec probabilité d, que le visite arrive en ayant suivi un lien depuis un nœud entrant
- on voit tout de suite le lien avec la marche aléatoire vue auparavant, à laquelle on ajoute deux  choses : un facteur $d$ qui pondère la somme, et un terme de téléportation uniforme $\frac{1-d}{N}$ qui prend la probabilité manquante. Quand d → 1, le terme de téléportation disparaît et on retrouve exactement la formule de la distribution stationnaire.
- 
La **téléportation** garantit que le visiteur peut toujours atteindre n'importe quel nœud, même depuis un dangling node ou une composante isolée, ce qui assure la convergence.

> **Valeur historique :** Brin & Page (1998) proposent d = 0.85, correspondant à l'hypothèse qu'un internaute suit des liens 85 % du temps et tape une URL « au hasard » 15 % du temps.

## 2.2 Gestion des dangling nodes

Un dangling node u (out_degree = 0) n'envoie aucun visiteur (selon aucune probabilité) à ses voisins via les liens.
Sans correction, la somme des PR ne vaut plus 1 après chaque itération (la probabilité « disparaît » dans u). Un peu comme un trou noir (ou un puit sans fond, ou un siphon…) qui « grignoterait » la probabilité d’en sortir ou d’y échapper. 

Plus rigoureusement, voici ce qu’il se passe : si le surfeur se trouve en u à l'instant t (avec probabilité p(t)[u] > 0), la règle de transition ne sait pas où l'envoyer à t+1 car il n'y a aucun arc sortant. La formule de mise à jour ne reçoit donc aucune contribution de u pour aucun voisin. La probabilité p(t)[u] est simplement ignorée dans le calcul de p(t+1), ce qui fait que $\sum_{v \in V} p_{t+1}(v) < 1$: le vecteur n'est plus une distribution de probabilité valide.

La correction PageRank restaure cette propriété en décidant que depuis un dangling node, le surfeur se téléporte uniformément, ce qui revient à ajouter des arcs virtuels vers tous les autres nœuds.

La correction standard consiste à redistribuer le PR des dangling nodes **uniformément** vers tous les nœuds, comme si le dangling node avait des arcs vers tout le monde :

$$
\text{contribution\_dangling}(v) = d \times \frac{\displaystyle\sum_{\substack{u \in V \\ \text{out\_degree}(u) = 0}} \text{PR}(u)}{N}
$$

La formule complète devient :

$$
\text{PR}(v) = \frac{1-d}{N} + d \left( \sum_{\substack{u \in V \\ u \rightarrow v}} \frac{\text{PR}(u)}{\text{out\_degree}(u)} + \frac{1}{N} \sum_{\substack{u \in V \\ \text{out\_degree}(u) = 0}} \text{PR}(u) \right)
$$

où :

$$
\text{dangling\_sum} = \sum_{\substack{u \in V \\ \text{out\_degree}(u) = 0}} \text{PR}(u)
$$

## 2.3 Algorithme par itération de puissance (le retour des valeurs propres)

### Le problème à résoudre
On a une jolie formule qui fait intervenir PR, et on sait que PR est la distribution stationnaire de la marche aléatoire avec téléportation. On sait qu'elle satisfait l'équation de point fixe :
$$
\text{PR} = T \times \text{PR}
$$
Mais comment calculer concrètement ce vecteur PR ?

L'équation `PR = T × PR` est un problème aux valeurs propres : PR est le vecteur propre de T associé à la valeur propre 1. 

Keuman ? revoyons l’action au ralenti :

> **Rappel sur ce qu'est un vecteur propre**
De manière générale, un vecteur propre d'une matrice M est un vecteur v qui, quand on lui applique M, donne le même vecteur à un facteur multiplicatif près :
$$
M \times \mathbf{v} = \lambda \times \mathbf{v}
$$
où λ est la valeur propre associée. Appliquer M à v ne change pas sa direction, ça l'étire ou le compresse d'un facteur λ, c'est tout.

>**Le cas particulier λ = 1**
Quand λ = 1, l'équation devient :
$$
M \times \mathbf{v} = \mathbf{v}
$$
>Appliquer M ne change rien du tout au vecteur. C'est un point fixe de la transformation M.

>**Le lien avec PageRank**
La matrice de transition T encode les probabilités de déplacement du vistieur. Appliquer T à un vecteur d'état p(t), c'est simuler un pas de la marche aléatoire :
$$
p_{(t+1)} = T \times p_{(t)}
$$
La distribution stationnaire PR est précisément le vecteur qui ne change plus quand on lui applique T, c'est-à-dire :
$$
T \times \text{PR} = \text{PR}
$$
C'est exactement la définition d'un vecteur propre associé à la valeur propre 1. PR est le vecteur que la matrice T "laisse en place".

Une première idée serait de résoudre ce système linéaire directement, c'est-à-dire inverser (I - T) pour trouver PR. C'est mathématiquement correct, mais inutilisable en pratique : sur un graphe web avec des milliards de nœuds, stocker et inverser une matrice T de taille N × N est hors de portée en mémoire comme en temps de calcul, d’autant qu’on a déjà vu que l’inversion de matrice est un problème vraiment complexe et coûteux en calcul.

Il faut donc une approche itérative qui ne manipule jamais la matrice T explicitement, mais seulement des vecteurs de taille N.

> La solution : simuler la marche jusqu'à stabilisation

L'idée est simple : puisque PR est l'état vers lequel la marche aléatoire converge à long terme, il suffit de simuler cette marche depuis une distribution initiale quelconque et d'observer où elle se stabilise.
On part d'une distribution uniforme PR[v] = 1/N, on considère que chaque nœud a la même probabilité initiale, et on applique répétitivement la formule PageRank jusqu'à ce que le vecteur ne change plus significativement d'une itération à l'autre. À ce stade, on a trouvé le point fixe : T × PR = PR, c'est-à-dire PR lui-même.
C'est ce qu'on appelle l'itération de puissance : appliquer T répétitivement "efface" progressivement toute l'information du vecteur initial, et ne laisse subsister que la composante associée à la valeur propre dominante qui est précisément PR.

```
PageRank(G, d, ε) :

  (1) Initialisation
     pour tout v : PR[v] ← 1/N        ← distribution uniforme

  (2) Itération
     répéter :
        dangling_sum ← Σ_{u : out_degree(u)=0} PR[u]

        pour tout v :
           link_sum ← Σ_{u→v} PR[u] / out_degree(u)
           PR_new[v] ← (1-d)/N  +  d × (link_sum + dangling_sum/N)

        delta ← max_v |PR_new[v] - PR[v]|   ← erreur de convergence
        PR ← PR_new

     jusqu'à delta < ε

  (3) Retourner PR
```

### Points délicats du pseudo-code

**(1) Initialisation à 1/N, pas à 0**
On part d'une distribution uniforme, pas du vecteur nul. Partir de 0 bloquerait
la convergence car T × 0 = 0 : on resterait au vecteur nul.

**(2) Ordre de mise à jour**
Le calcul de `PR_new[v]` utilise les valeurs de l'**ancienne** itération PR[u], pas
les nouvelles. C'est une mise à jour **synchrone**
En pratique : calculer tous les PR_new[v] avant d'écraser PR.

**(2) `dangling_sum` recalculé à chaque itération**
Les scores PR[u] des dangling nodes changent à chaque itération. Il faut donc
recalculer `dangling_sum` à chaque tour de boucle, pas une seule fois en amont.

**(2) Précalculer les listes de voisins entrants**
Pour chaque v, on parcourt `in_nodes[v]`, la liste des u tels que u→v.
Précalculer cette structure une fois avant la boucle est essentiel pour éviter
de reconstruire les voisins entrants à chaque itération (coût O(E) par itération
au lieu de O(V×E)).

**(3) La somme des PR vaut toujours 1**
Si l'implémentation est correcte, `Σ_v PR[v] == 1` après chaque itération.
C'est une propriété utile pour détecter des bugs (accumulation ou perte de probabilité).

# PARTIE 3 : Facteur d'amortissement et convergence

## 3.1 Rôle de d

Le facteur d'amortissement d contrôle l'équilibre entre deux comportements opposés :

| | d proche de 0 | d proche de 1 |
|---|---|---|
| Comportement | Quasi-téléportation pure | Quasi-marche aléatoire pure |
| Scores | Presque uniformes (≈ 1/N) | Très concentrés sur les hubs |
| Sensibilité à la structure | Faible | Forte |
| Vitesse de convergence | Rapide | Lente |

Avec d = 0 : PR[v] = 1/N pour tout v, la structure du graphe est ignorée.
Avec d = 1 : la téléportation disparaît, les problèmes de dangling nodes et de déconnexion réapparaissent et la convergence n'est plus garantie.

**d = 0.85 est un compromis empirique**, pas un résultat théorique. Pour des graphes très différents du web (graphes de citations, réseaux sociaux internes, pipelines), il est légitime d'ajuster ce paramètre.

## 3.2 Convergence

La convergence est garantie dès lors que la téléportation est active (d < 1) : elle assure que le visiteur peut toujours atteindre n'importe quel nœud depuis n'importe quel autre, ce qui rend la distribution stationnaire unique et indépendante du point de départ.
La vitesse de convergence dépend de d : plus d est petit, plus la téléportation est fréquente, plus la convergence est rapide. Avec d = 0.85, 50 à 100 itérations suffisent en pratique pour la plupart des graphes.

## 3.3 Critère d'arrêt

On mesure la convergence par l'**erreur max** entre deux itérations consécutives :

```
delta = max_v |PR_new[v] - PR[v]|
```

On s'arrête quand `delta < ε`. Le choix de ε dépend de l'usage :
- ε = 1e-6 suffit généralement pour un ranking (l'ordre relatif est stable)
- ε = 1e-10 pour des calculs nécessitant une précision numérique fine

Une alternative est l'erreur L1 : `Σ_v |PR_new[v] - PR[v]|`. Elle est plus robuste sur les grands graphes où l'erreur max peut être dominée par un seul nœud atypique.

![Convergence d](./Images/Convergence-d_PageRank.png)

# PARTIE 4 : Random Walk with Restart (RWR)

## 4.1 Motivation : de l'importance globale à la proximité personnalisée

PageRank calcule l'importance **globale** d'un nœud, indépendante de tout point de départ.
C'est utile pour un moteur de recherche qui veut scorer toutes les pages du web de façon universelle.

En revanche, pour un **système de recommandation**, on veut quelque chose de différent : l'importance d'un nœud **relative à un nœud source spécifique** (le profil d'un utilisateur, un item consulté, une entité dans un knowledge graph).

**Random Walk with Restart (RWR)** répond à cette question. La seule modification par rapport à PageRank : au lieu de se téléporter vers un nœud **uniformément aléatoire**, le visiteur retourne toujours au **nœud source** :

```
PageRank :  avec proba (1-d) → nœud aléatoire parmi tous les nœuds
RWR :       avec proba (1-d) → retour au nœud source (toujours)
```

## 4.2 Formule RWR

Le vecteur RWR[· | s] donne le **score de proximité** de chaque nœud par rapport à s. Un score élevé signifie que le visiteur partant de s se retrouve fréquemment sur ce nœud.

```
RWR[v | s] = (1-d) × 1{v == s}   +   d × Σ_{u→v}  RWR[u | s] / out_degree(u)
              ──────────────────         ────────────────────────────────────
              restart vers s             contribution des liens entrants
              (seulement si v = s)
```
$$
\text{RWR}(v \mid s) = (1-d) \cdot \mathbf{1}_{[v = s]} + d \sum_{\substack{u \in V \\ u \rightarrow v}} \frac{\text{RWR}(u \mid s)}{\text{out\_degree}(u)}
$$

* Si v ≠ s :
$$
\text{RWR}(v \mid s) = O + d \sum_{\substack{u \in V \\ u \rightarrow v}} \frac{\text{RWR}(u \mid s)}{\text{out\_degree}(u)}
$$
    le score de v ne vient que des liens entrants, il n'y a aucune contribution du restart
* Si v = s :
$$
\text{RWR}(s \mid s) = (1-d) + d \sum_{\substack{u \in V \\ u \rightarrow v}} \frac{\text{RWR}(u \mid s)}{\text{out\_degree}(u)}
$$
    le score de s reçoit en plus le terme (1-d), qui représente la probabilité de restart

Le nœud source bénéficie du terme de restart. Tous les autres nœuds n'ont comme source de score que les liens qui pointent vers eux.
C'est la différence fondamentale avec PageRank où le terme (1-d)/N est distribué uniformément sur tous les nœuds, chaque nœud reçoit un peu de probabilité de téléportation. Ici toute cette probabilité de restart est concentrée sur s.

## 4.3 Algorithme

L'algorithme est identique à PageRank, avec une seule différence dans la téléportation :

```
RWR(G, source s, d, ε) :

  (1) Initialisation
     pour tout v : RWR[v] ← 1/N

  (2) Itération
     répéter :
        pour tout v :
           link_sum ← Σ_{u→v} RWR[u] / out_degree(u)

           si v == s :
              RWR_new[v] ← (1-d)  +  d × link_sum   ← restart concentré sur s
           sinon :
              RWR_new[v] ←  0     +  d × link_sum

        delta ← max_v |RWR_new[v] - RWR[v]|
        RWR ← RWR_new

     jusqu'à delta < ε

  (3) Retourner RWR[· | s]
```

### Points délicats

**La somme des RWR vaut toujours 1**
Même remarque que pour PageRank : c'est une invariant utile pour valider l'implémentation.

**Pas de correction pour les dangling nodes ?**
Avec RWR, on peut choisir de rediriger les dangling nodes vers la source plutôt que vers tous les nœuds uniformément. Le choix dépend du contexte applicatif.

**Un appel par nœud source**
RWR doit être recalculé pour chaque nœud source. Sur un grand graphe, cela peut être coûteux. Des techniques matricielles (inversion de (I - d×T)) permettent de calculer tous les RWR simultanément, au prix d'une complexité spatiale O(V²).

## 4.4 Application : graphe bipartite et recommandation collaborative

Le cas d'usage le plus direct en data science est la **recommandation par graphe**.

On construit un **graphe bipartite** utilisateurs × items :
- Un nœud par utilisateur, un nœud par item
- Un arc U → I si l'utilisateur U a interagi positivement avec l'item I
- Un arc I → U en retour (graphe non orienté converti en orienté)

```
   U1 → I1
   U1 → I2      (U1 a aimé I1, I2, I3)
   U1 → I3
   U2 → I2      (U2 a aimé I2, I4)
   U2 → I4
   U3 → I3      (U3 a aimé I3, I4, I5)
   U3 → I4
   U3 → I5
   + arcs retour Ix → Ux pour chaque interaction
```

**RWR depuis U1** va :
1. Remonter vers les items qu'U1 a aimés (I1, I2, I3), d’où un  score élevé
2. De ces items, atteindre les autres utilisateurs qui les ont aussi aimés (U2 via I2, U3 via I3)
3. De ces utilisateurs similaires, atteindre les items qu'ils ont aimés mais qu'U1 n'a pas vus (I4 via U2 et U3, I5 via U3)

Les items non vus avec le score RWR le plus élevé sont les recommandations.

C'est le principe du **filtrage collaboratif par graphe** : on n'a pas besoin de définir explicitement une similarité entre utilisateurs car c’est la structure du graphe la capture implicitement via les chemins de la marche aléatoire.

# PARTIE 5 : Pour aller plus loin

Cette séance 4 est probablement la plus importante de tout ce module. Elle couvre l'ensemble des algorithmes de graphes fondamentaux utilisés en data engineering et en data science. Les tableaux ci-dessous récapitulent tout ce qui a été vu dans les trois notebooks et comment chaque concept se connecte à des problèmes réels.

## Notebook A : Graphes, BFS, DFS, Backtracking, DAG

| Concept | Pour quoi faire ? | Application en data |
|---|---|---|
| **Liste d'adjacence** | Représentation mémoire efficace O(V+E) | Tous les frameworks graphes : NetworkX, Neo4j, DGL |
| **BFS** | Plus court chemin non pondéré, exploration par niveaux | Recherche de voisins dans un réseau social, crawling web |
| **DFS** | Exploration exhaustive, détection de cycle | Analyse de dépendances, parseurs, moteurs de règles |
| **Backtracking** | Exploration contrainte avec retour arrière | Optimisation combinatoire, résolution de contraintes (scheduling) |
| **Détection de cycle** | Vérifier qu'un graphe est un DAG | Validation d'un pipeline Airflow avant déploiement |
| **Tri topologique (Kahn)** | Ordonner les tâches selon leurs dépendances | Apache Airflow, dbt, Spark DAG scheduler |
| **Tri topologique (DFS)** | Variante récursive du tri topo | Compilateurs, résolution de dépendances (npm, pip) |
| **Propagation d'impact** | Identifier les nœuds en aval d'un changement | Alertes pipeline, invalidation de cache, CI/CD |


## Notebook B : Plus courts chemins

| Concept | Pour quoi faier ? | Application en data |
|---|---|---|
| **Dijkstra** | Plus court chemin pondéré, poids ≥ 0 | Routage réseau, GPS, graphes de similarité |
| **Lazy deletion** | Pattern `heapq` sans mise à jour in-place | Tout algorithme Python utilisant une file de priorité |
| **A\*** | Dijkstra guidé par heuristique, explore moins | Moteurs de jeu, navigation géospatiale, planification |
| **Heuristique admissible** | Ne surestime jamais → optimalité garantie | Distance Haversine pour réseaux routiers réels |


## Notebook C : PageRank et recommandation

| Concept | Pour quoi faire ?| Application en data |
|---|---|---|
| **Marche aléatoire** | Distribution stationnaire = importance des nœuds | Analyse de réseaux, détection de communautés |
| **PageRank** | Score d'importance global par itération de puissance | Moteurs de recherche, ranking d'entités dans un KG |
| **Facteur d'amortissement** | Équilibre structure / téléportation | Paramètre clé à régler selon la densité du graphe |
| **RWR** | Importance relative à un nœud source | Recommandation collaborative, détection d'anomalies |

## Arbre de décision pour choisir le bon algorithme


Mon graphe est-il pondéré ?

- Non → BFS  (plus court en nb d'arêtes, O(V+E))
- Oui, poids ≥ 0
  - Je connais la cible et j'ai une heuristique admissible → A*
  - Sinon → Dijkstra 

- Je veux ordonner des tâches avec dépendances  → Tri topologique (Kahn ou DFS)
- Je veux détecter un cycle dans un graphe orienté → DFS (white, gray, black)
- Je veux scorer l'importance globale des nœuds  → PageRank
- Je veux scorer la proximité à un nœud source   → RWR (Random Walk with Restart)
- Je veux explorer exhaustivement sous contraintes → Backtracking

## Comment exploiter tout ça en data science

### Pipelines de données (DAG)
Tout pipeline de traitement est un DAG. Maîtriser le tri topologique permet de comprendre comment Airflow planifie les tâches, pourquoi dbt ordonne les modèles SQL, et comment Spark construit son plan d'exécution physique. La détection de cycle est la première validation à faire avant tout déploiement.

### Feature engineering sur graphes
Les propriétés de graphe (degré, PageRank, centralité, composante connexe) sont des **features puissantes** pour les modèles ML sur des données relationnelles : détection de fraude, scoring de crédit, prédiction de lien dans un réseau social.
Ces features s'extraient directement avec NetworkX ou PyG (PyTorch Geometric).

### Systèmes de recommandation
BFS, RWR et PageRank sont à la base des moteurs de recommandation par graphe.
Le graphe bipartite utilisateurs × items est la représentation naturelle des données d'interaction (clics, achats, notes). RWR donne des recommandations personnalisées sans nécessiter de factorisation matricielle.

### Recherche de similarité (ANN)
Dijkstra et A* se généralisent aux **index de recherche approximative** (ANN).
HNSW (*Hierarchical Navigable Small World*) construit un graphe de proximité multi-couches et utilise un BFS guidé (proche de A*) pour trouver les k voisins les plus proches dans un espace vectoriel de haute dimension.
C'est le moteur de Pinecone, Weaviate, pgvector, et des bases vectorielles modernes.

### Knowledge Graphs
Les bases de connaissance (Wikidata, Google KG, bases métier) sont des graphes orientés pondérés. PageRank, BFS de proximité et traversées DFS y servent pour l'inférence de relations, la complétion de graphe et le question answering.

## Références pour aller plus loin

- **NetworkX** : bibliothèque Python de référence pour les graphes : `pip install networkx`
- **PyTorch Geometric (PyG)** : graphes et GNN : `https://pyg.org`
- **OSMnx** : graphes routiers réels depuis OpenStreetMap : `pip install osmnx`
- **Brin & Page (1998)** : article original PageRank : *The Anatomy of a Large-Scale Hypertextual Web Search Engine*
- **Tong, Faloutsos & Pan (2006)** : article original RWR : *Fast Random Walk with Restart and Its Applications*
- **HNSW (Malkov & Yashunin, 2018)** : *Efficient and robust approximate nearest neighbor search using HNSW graphs*

## Sujets de mini-mémoire : récapitulatif

Ces éléments ne sont pas traités dans ce notebook. Ils constituent des pistes d'approfondissement pour les étudiants qui travaillent sur un mini-mémoire en lien avec cette séance.

- **Sujet 4 : Graphes et recommandation** Graphe bipartite, RWR, PageRank (cf. Notebook C).Implémentation RWR from scratch, évaluation précision@K, rappel@K
- **Sujet 5 : Dijkstra, A * et géospatial** Dijkstra, A*, heuristique admissible (Notebook B), OSMnx, Haversine, Folium, comparaison nb nœuds explorés
- **À appronfondir : KD-trees et ANN** BFS guidé, A* (Notebooks A et B), Benchmark dimension croissante, HNSW, FAISS 